# Dask Chunk Grid Analysis

This notebook summarizes CSV output from `benchmarks/scripts/dask_chunk_grid.py` for issue #2036. It is intentionally small: run the benchmark script first, then use the tables and plots below to compare on-disk chunks, virtual Dask chunks, worker settings, task counts, storage overhead, and coarse memory readings.

In [ ]:
from pathlib import Path

import pandas as pd

results_path = Path("../results/dask_chunk_grid.csv")
df = pd.read_csv(results_path)
df.head()

In [ ]:
metadata_cols = [
    "run_started_at",
    "python_version",
    "platform",
    "anndata_version",
    "numpy_version",
    "h5py_version",
    "zarr_version",
    "dask_version",
    "distributed_version",
    "scanpy_version",
]

df[metadata_cols].drop_duplicates()

In [ ]:
group_cols = [
    "store_type",
    "zarr_format",
    "zarr_shards",
    "shape",
    "on_disk_chunks",
    "dask_chunks_arg",
    "workers",
    "threads_per_worker",
    "processes",
    "workload",
]

summary = (
    df.groupby(group_cols, dropna=False)
    .agg(
        elapsed_median_s=("elapsed_s", "median"),
        elapsed_min_s=("elapsed_s", "min"),
        task_count=("task_count", "median"),
        dataset_nbytes=("dataset_nbytes", "median"),
        store_nbytes=("store_nbytes", "median"),
        worker_rss_after_mb=("worker_rss_after_mb", "median"),
        runs=("elapsed_s", "size"),
    )
    .reset_index()
    .sort_values(["workload", "elapsed_median_s"])
)
summary["store_overhead_ratio"] = summary["store_nbytes"] / summary["dataset_nbytes"]
summary.head(20)

In [ ]:
best_by_workload = summary.loc[
    summary.groupby(["store_type", "workload"])["elapsed_median_s"].idxmin()
]
best_by_workload.sort_values(["store_type", "workload"])

In [ ]:
baseline_cols = [
    "store_type",
    "zarr_format",
    "zarr_shards",
    "shape",
    "on_disk_chunks",
    "workers",
    "threads_per_worker",
    "processes",
    "workload",
]

baseline = summary.loc[summary["dask_chunks_arg"] == "default", baseline_cols + ["elapsed_median_s"]]
baseline = baseline.rename(columns={"elapsed_median_s": "default_elapsed_median_s"})
speedups = summary.merge(baseline, on=baseline_cols, how="left")
speedups["speedup_vs_default"] = speedups["default_elapsed_median_s"] / speedups["elapsed_median_s"]
speedups.sort_values(["workload", "speedup_vs_default"], ascending=[True, False]).head(30)

In [ ]:
import matplotlib.pyplot as plt

for workload, workload_df in summary.groupby("workload"):
    fig, ax = plt.subplots(figsize=(10, 5))
    labels = []
    for label, label_df in workload_df.groupby(
        ["store_type", "on_disk_chunks", "workers", "threads_per_worker"]
    ):
        label_df = label_df.sort_values("dask_chunks_arg")
        labels.append(str(label))
        ax.plot(
            label_df["dask_chunks_arg"],
            label_df["elapsed_median_s"],
            marker="o",
            label=str(label),
        )
    ax.set_title(workload)
    ax.set_xlabel("Dask chunks argument")
    ax.set_ylabel("Median elapsed seconds")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize="small", bbox_to_anchor=(1.05, 1), loc="upper left")
    fig.tight_layout()
    plt.show()